In [1]:
import pandas as pd
pd.__version__


'3.0.0'

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
import random

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
random.seed(42)

n = 50

# --- USERS RAW CSV ---
users_raw_df = pd.DataFrame({
    "user_id": [f"U{1000+i}" for i in range(n)] + [None],   # missing user_id
    "status": np.random.choice(["Active", "ACTIVE ", " inactive", "Cancelled"], n+1),
    "signup_date": [
        (datetime.today() - timedelta(days=random.randint(30, 900))).strftime("%Y-%m-%d")
        for _ in range(n+1)
    ],
    "monthly_fee": np.random.choice([9.99, 14.99, 19.99, None], n+1),
    "months_active": np.random.randint(1, 24, n+1),
    "age": np.random.choice([18, 25, 30, "unknown", None], n+1),
    "country_code": np.random.choice(["AU", "US", "IN", "GB", None], n+1)
})

users_raw_df.to_csv(RAW_DIR / "users_raw.csv", index=False)

# --- SUPPORT RAW JSON ---
support = []
valid_ids = users_raw_df["user_id"].dropna().astype(str).tolist()

for i in range(80):
    support.append({
        "ticket_id": f"T{i+1}",
        "user_id": random.choice(valid_ids),
        "issue_type": random.choice(["billing", "technical", "account"]),
        "created_at": (datetime.today() - timedelta(days=random.randint(1, 365))).isoformat()
    })

with open(RAW_DIR / "support_raw.json", "w", encoding="utf-8") as f:
    json.dump(support, f, indent=2)

"Raw CSV and JSON created"


'Raw CSV and JSON created'

In [3]:
from pathlib import Path
list(Path("../data/raw").glob("*"))


[WindowsPath('../data/raw/support_raw.json'),
 WindowsPath('../data/raw/users_raw.csv')]

In [4]:
## 3. Cleaning & Validation


In [5]:
import pandas as pd
import json

users = pd.read_csv("../data/raw/users_raw.csv")

with open("../data/raw/support_raw.json", "r", encoding="utf-8") as f:
    support_json = json.load(f)
support = pd.json_normalize(support_json)

users.head(), support.head()


(  user_id     status signup_date  monthly_fee  months_active  age country_code
 0   U1000   inactive  2024-03-27        14.99             12   30           IN
 1   U1001  Cancelled  2025-09-18        19.99              2   30           IN
 2   U1002     Active  2025-12-16          NaN             10   18           AU
 3   U1003   inactive  2023-12-13        19.99              4   30          NaN
 4   U1004   inactive  2025-04-04          NaN             14  NaN           AU,
   ticket_id user_id issue_type                  created_at
 0        T1   U1006  technical  2025-08-16T20:46:58.576100
 1        T2   U1038  technical  2026-01-17T20:46:58.576100
 2        T3   U1046  technical  2025-05-10T20:46:58.576100
 3        T4   U1007  technical  2025-12-30T20:46:58.576100
 4        T5   U1035  technical  2025-03-24T20:46:58.576100)

In [6]:
users["status"] = users["status"].astype(str).str.strip().str.lower()

users["status"].value_counts(dropna=False)


status
active       21
cancelled    16
inactive     14
Name: count, dtype: int64

In [7]:
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")

users.dtypes


user_id                     str
status                      str
signup_date      datetime64[us]
monthly_fee             float64
months_active             int64
age                         str
country_code                str
dtype: object

In [8]:
import numpy as np

users["monthly_fee"] = pd.to_numeric(users["monthly_fee"], errors="coerce")

median_fee = users["monthly_fee"].median()
users["monthly_fee"] = users["monthly_fee"].fillna(median_fee)

median_fee, users["monthly_fee"].isna().sum()


(np.float64(14.99), np.int64(0))

In [9]:
users["age"] = pd.to_numeric(users["age"], errors="coerce")

users["age"].dtype, users["age"].isna().sum()


(dtype('float64'), np.int64(23))

In [10]:
# Count issues before dropping
missing_user_id = users["user_id"].isna().sum()
dup_user_id = users["user_id"].duplicated().sum()

missing_user_id, dup_user_id


(np.int64(1), np.int64(0))

In [11]:
users = users.dropna(subset=["user_id"])

# Verify
users["user_id"].isna().sum(), users["user_id"].duplicated().sum(), users.shape


(np.int64(0), np.int64(0), (50, 7))

In [12]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_path = PROCESSED_DIR / "users_cleaned.csv"
users.to_csv(clean_path, index=False)

str(clean_path)


'..\\data\\processed\\users_cleaned.csv'

In [13]:
## 4. Feature Engineering


In [14]:
users["is_active_subscription"] = (users["status"] == "active").astype(int)
users["is_active_subscription"].value_counts()


is_active_subscription
0    29
1    21
Name: count, dtype: int64

In [15]:
users["months_active"] = pd.to_numeric(users["months_active"], errors="coerce").fillna(0)

users["revenue_estimate"] = users["monthly_fee"] * users["months_active"]

users[["monthly_fee", "months_active", "revenue_estimate"]].head()


,monthly_fee,months_active,revenue_estimate
0,14.99,12,179.88
1,19.99,2,39.98
2,14.99,10,149.90
3,19.99,4,79.96
4,14.99,14,209.86


In [17]:
users["subscription_length_days"] = users["months_active"] * 30

users[["months_active", "subscription_length_days"]].head()


,months_active,subscription_length_days
0,12,360
1,2,60
2,10,300
3,4,120
4,14,420


In [18]:
from pathlib import Path

FINAL_DIR = Path("../data/processed")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_path = FINAL_DIR / "users_analytics_ready.csv"
users.to_csv(final_path, index=False)

str(final_path)


'..\\data\\processed\\users_analytics_ready.csv'

In [19]:
summary = {
    "total_users": len(users),
    "active_users": int(users["is_active_subscription"].sum()),
    "inactive_users": int((users["is_active_subscription"] == 0).sum()),
    "total_estimated_revenue": round(users["revenue_estimate"].sum(), 2),
    "avg_revenue_per_user": round(users["revenue_estimate"].mean(), 2),
}

summary


{'total_users': 50,
 'active_users': 21,
 'inactive_users': 29,
 'total_estimated_revenue': np.float64(7944.68),
 'avg_revenue_per_user': np.float64(158.89)}

In [20]:
import requests
import time

def fetch_country_name(code, retries=3, backoff=1):
    """
    Fetch country name from REST Countries API.
    Returns None if API fails.
    """
    if pd.isna(code):
        return None

    code = str(code).strip().upper()
    url = f"https://restcountries.com/v3.1/alpha/{code}"

    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=5)

            if response.status_code == 200:
                data = response.json()
                if isinstance(data, list) and len(data) > 0:
                    return data[0]["name"]["common"]
                return None

            # Retry on server / rate-limit issues
            if response.status_code in [429, 500, 502, 503, 504]:
                time.sleep(backoff)

        except requests.RequestException:
            time.sleep(backoff)

    return None


In [21]:
# Get unique country codes
country_codes = users["country_code"].dropna().unique()
country_codes


<StringArray>
['IN', 'AU', 'US', 'GB']
Length: 4, dtype: str

In [22]:
country_lookup = {}

for code in country_codes:
    country_lookup[code] = fetch_country_name(code)

country_lookup


{'IN': 'India',
 'AU': 'Australia',
 'US': 'United States',
 'GB': 'United Kingdom'}

In [23]:
users["country_name"] = users["country_code"].map(country_lookup)

users[["country_code", "country_name"]].drop_duplicates()


,country_code,country_name
0,IN,India
2,AU,Australia
3,NaN,NaN
6,US,United States
7,GB,United Kingdom


In [24]:
from pathlib import Path

ENRICHED_PATH = Path("../data/processed/users_enriched.csv")
users.to_csv(ENRICHED_PATH, index=False)

str(ENRICHED_PATH)


'..\\data\\processed\\users_enriched.csv'

In [25]:
from sqlalchemy import create_engine

# SQLite database stored locally
engine = create_engine("sqlite:///../data/db/streamsmart.db")

engine


Engine(sqlite:///../data/db/streamsmart.db)

In [26]:
from sqlalchemy import (
    Table, Column, String, Integer, Float, DateTime, MetaData
)

metadata = MetaData()

users_table = Table(
    "users",
    metadata,
    Column("user_id", String, primary_key=True),
    Column("status", String),
    Column("age", Float),
    Column("country_code", String),
    Column("country_name", String),
    Column("signup_date", DateTime),
)

metadata.create_all(engine)

users_table


OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [27]:
from pathlib import Path

DB_DIR = Path("../data/db")
DB_DIR.mkdir(parents=True, exist_ok=True)

DB_DIR


WindowsPath('../data/db')

In [28]:
from sqlalchemy import (
    Table, Column, String, Float, DateTime, MetaData
)

metadata = MetaData()

users_table = Table(
    "users",
    metadata,
    Column("user_id", String, primary_key=True),
    Column("status", String),
    Column("age", Float),
    Column("country_code", String),
    Column("country_name", String),
    Column("signup_date", DateTime),
)

metadata.create_all(engine)

users_table


Table('users', MetaData(), Column('user_id', String(), table=<users>, primary_key=True, nullable=False), Column('status', String(), table=<users>), Column('age', Float(), table=<users>), Column('country_code', String(), table=<users>), Column('country_name', String(), table=<users>), Column('signup_date', DateTime(), table=<users>), schema=None)

In [29]:
from sqlalchemy import insert

# Select only user-level columns
users_to_insert = users[[
    "user_id",
    "status",
    "age",
    "country_code",
    "country_name",
    "signup_date"
]]

# Insert rows
with engine.begin() as conn:
    conn.execute(
        insert(users_table),
        users_to_insert.to_dict(orient="records")
    )

"users inserted"


'users inserted'

In [30]:
from sqlalchemy import (
    Table, Column, Integer, String, Float, MetaData, ForeignKey
)

metadata = MetaData()

subscriptions_table = Table(
    "subscriptions",
    metadata,
    Column("subscription_id", Integer, primary_key=True, autoincrement=True),
    Column("user_id", String, ForeignKey("users.user_id")),
    Column("months_active", Integer),
    Column("monthly_fee", Float),
    Column("revenue_estimate", Float),
    Column("is_active_subscription", Integer),
    Column("subscription_length_days", Integer),
)

metadata.create_all(engine)

subscriptions_table


NoReferencedTableError: Foreign key associated with column 'subscriptions.user_id' could not find table 'users' with which to generate a foreign key to target column 'user_id'

In [31]:
from sqlalchemy import (
    Table, Column, Integer, String, Float, MetaData, ForeignKey
)

# IMPORTANT: reuse the SAME metadata
subscriptions_table = Table(
    "subscriptions",
    metadata,   # <-- reuse existing metadata
    Column("subscription_id", Integer, primary_key=True, autoincrement=True),
    Column("user_id", String, ForeignKey("users.user_id")),
    Column("months_active", Integer),
    Column("monthly_fee", Float),
    Column("revenue_estimate", Float),
    Column("is_active_subscription", Integer),
    Column("subscription_length_days", Integer),
)

metadata.create_all(engine)

subscriptions_table


InvalidRequestError: Table 'subscriptions' is already defined for this MetaData instance.  Specify 'extend_existing=True' to redefine options and columns on an existing Table object.

In [32]:
# Reuse the already-registered subscriptions table
subscriptions_table = metadata.tables["subscriptions"]

subscriptions_table


Table('subscriptions', MetaData(), Column('subscription_id', Integer(), table=<subscriptions>, primary_key=True, nullable=False), Column('user_id', String(), ForeignKey('users.user_id'), table=<subscriptions>), Column('months_active', Integer(), table=<subscriptions>), Column('monthly_fee', Float(), table=<subscriptions>), Column('revenue_estimate', Float(), table=<subscriptions>), Column('is_active_subscription', Integer(), table=<subscriptions>), Column('subscription_length_days', Integer(), table=<subscriptions>), schema=None)

In [33]:
from sqlalchemy import insert

# Select subscription-level columns
subscriptions_to_insert = users[[
    "user_id",
    "months_active",
    "monthly_fee",
    "revenue_estimate",
    "is_active_subscription",
    "subscription_length_days"
]]

# Insert into subscriptions table
with engine.begin() as conn:
    conn.execute(
        insert(subscriptions_table),
        subscriptions_to_insert.to_dict(orient="records")
    )

"subscriptions inserted"


OperationalError: (sqlite3.OperationalError) no such table: subscriptions
[SQL: INSERT INTO subscriptions (user_id, months_active, monthly_fee, revenue_estimate, is_active_subscription, subscription_length_days) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: [('U1000', 12, 14.99, 179.88, 0, 360), ('U1001', 2, 19.99, 39.98, 0, 60), ('U1002', 10, 14.99, 149.9, 1, 300), ('U1003', 4, 19.99, 79.96, 0, 120), ('U1004', 14, 14.99, 209.86, 0, 420), ('U1005', 16, 14.99, 239.84, 0, 480), ('U1006', 15, 9.99, 149.85, 1, 450), ('U1007', 8, 19.99, 159.92, 1, 240)  ... displaying 10 of 50 total bound parameter sets ...  ('U1048', 3, 9.99, 29.97, 1, 90), ('U1049', 12, 19.99, 239.88, 1, 360)]]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [34]:
from sqlalchemy import MetaData

metadata = MetaData()
metadata.reflect(bind=engine)

metadata.tables.keys()


dict_keys(['users'])

In [35]:
from sqlalchemy import Table, Column, Integer, String, Float, ForeignKey

subscriptions_table = Table(
    "subscriptions",
    metadata,
    Column("subscription_id", Integer, primary_key=True, autoincrement=True),
    Column("user_id", String, ForeignKey("users.user_id")),
    Column("months_active", Integer),
    Column("monthly_fee", Float),
    Column("revenue_estimate", Float),
    Column("is_active_subscription", Integer),
    Column("subscription_length_days", Integer),
)

subscriptions_table.create(engine)

subscriptions_table


Table('subscriptions', MetaData(), Column('subscription_id', Integer(), table=<subscriptions>, primary_key=True, nullable=False), Column('user_id', String(), ForeignKey('users.user_id'), table=<subscriptions>), Column('months_active', Integer(), table=<subscriptions>), Column('monthly_fee', Float(), table=<subscriptions>), Column('revenue_estimate', Float(), table=<subscriptions>), Column('is_active_subscription', Integer(), table=<subscriptions>), Column('subscription_length_days', Integer(), table=<subscriptions>), schema=None)

In [38]:
from sqlalchemy import insert

subscriptions_to_insert = users[[
    "user_id",
    "months_active",
    "monthly_fee",
    "revenue_estimate",
    "is_active_subscription",
    "subscription_length_days"
]]

with engine.begin() as conn:
    conn.execute(
        insert(subscriptions_table),
        subscriptions_to_insert.to_dict(orient="records")
    )

"subscriptions inserted"


'subscriptions inserted'

In [39]:
from sqlalchemy import Table, Column, Integer, Float, DateTime
from datetime import datetime

analytics_table = Table(
    "analytics_summary",
    metadata,
    Column("run_id", Integer, primary_key=True, autoincrement=True),
    Column("run_timestamp", DateTime),
    Column("total_users", Integer),
    Column("active_users", Integer),
    Column("total_estimated_revenue", Float),
)

analytics_table.create(engine)

analytics_table


Table('analytics_summary', MetaData(), Column('run_id', Integer(), table=<analytics_summary>, primary_key=True, nullable=False), Column('run_timestamp', DateTime(), table=<analytics_summary>), Column('total_users', Integer(), table=<analytics_summary>), Column('active_users', Integer(), table=<analytics_summary>), Column('total_estimated_revenue', Float(), table=<analytics_summary>), schema=None)

In [40]:
from sqlalchemy import insert
from datetime import datetime

summary_record = {
    "run_timestamp": datetime.utcnow(),
    "total_users": len(users),
    "active_users": int(users["is_active_subscription"].sum()),
    "total_estimated_revenue": float(users["revenue_estimate"].sum()),
}

with engine.begin() as conn:
    conn.execute(
        insert(analytics_table),
        [summary_record]
    )

"analytics summary inserted"


'analytics summary inserted'

In [41]:
engine.table_names()


AttributeError: 'Engine' object has no attribute 'table_names'

In [42]:
from sqlalchemy import inspect

inspector = inspect(engine)
inspector.get_table_names()


['analytics_summary', 'subscriptions', 'users']

### Q1. Dropping records vs imputing values

Dropping records is appropriate when a field is critical to business logic or uniqueness, such as a missing `user_id`, because keeping it would break joins, create duplicates, or corrupt downstream analysis. In these cases, imputing would introduce false entities and reduce data integrity.

Imputation is appropriate when the missing value is non-identifying and the record is otherwise valid, such as `monthly_fee`. Here, using the median preserves population-level patterns while avoiding unnecessary data loss.

The decision depends on business impact: if incorrect values are more harmful than fewer records, drop; if losing records harms representativeness more than estimation error, impute.


### Q2. Assumptions and bias in revenue estimation

The revenue estimate assumes that users paid the same `monthly_fee` for every month in `months_active` and that there were no discounts, pauses, refunds, or price changes. It also assumes all recorded months represent fully billed periods.

These assumptions can bias conclusions by overstating revenue if users churned mid-month, received promotions, or downgraded plans. Conversely, revenue may be understated if users upgraded or paid additional fees not captured in the dataset. As a result, this estimate is suitable for high-level trend analysis but not for financial reporting or forecasting.
